# Модуль 4. От скоров к решениям: поиск бизнес-порога

**Длительность:** 90 минут
**Формат:** теория (65 мин) + практика (30 мин)
**Цель модуля:** Учащийся должен понять, что порог 0.5 — техническая конвенция, а не универсально верное решение; освоить формализацию задачи поиска порога через матрицу бизнес-издержек; уметь реализовать и обосновать алгоритм численного поиска оптимального порога, а также понимать границы применимости аналитического (байесовского) решения этой задачи.

## 1. Введение: от ранжирования к решению (5 мин)

### 1.1. Что мы умели до этого модуля
В Модулях 2–3 мы научились оценивать, насколько хорошо модель **ранжирует** объекты: ROC-AUC и PR-AUC отвечают на вопрос «в среднем, по всем возможным порогам, насколько разумно модель упорядочивает positive и negative объекты по скору». Обе метрики принципиально не зависят от конкретного порога — в этом их сила как инструмента сравнения моделей.

### 1.2. Проблема продакшена
Но в продакшене модель не выдает «в среднем хорошее ранжирование» — она принимает **одно конкретное решение по каждому конкретному объекту**: заблокировать транзакцию или пропустить, направить пациента на дополнительное обследование или нет, показать баннер или нет. Для этого единственного решения нужен единственный порог $t$, применяемый к вероятности $p = \hat{P}(y=1 \mid x)$, которую выдает модель.

Этот модуль — про то, как выбрать этот порог не произвольно, а обоснованно, минимизируя реальные издержки бизнеса.

## 2. Порог 0.5: конвенция, а не закон (10 мин)

### 2.1. Откуда берется 0.5 по умолчанию
Когда вызывается `model.predict(X)` в Scikit-Learn, под капотом происходит ровно следующее:

In [ ]:
y_pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)

Порог 0.5 зашит в реализацию по умолчанию не потому, что это математически доказанный оптимум для любой задачи, а потому что это простая, симметричная точка отсчета: «класс, вероятность которого выше — тот и предсказываем».

### 2.2. Когда 0.5 действительно оптимален
Порог 0.5 минимизирует **общее число ошибок** (не взвешенных по стоимости) при одном дополнительном условии: модель откалибрована, то есть её вероятности $p$ действительно отражают истинную частоту положительного исхода (подробно калибровка разбирается в Модуле 8). При этом условии 0.5 — это порог, при котором для каждого объекта мы выбираем класс с большей апостериорной вероятностью, что минимизирует **ожидаемое число неверных предсказаний**, если ошибка в любую сторону стоит одинаково.

### 2.3. Почему в реальных задачах это условие почти никогда не выполняется

Есть минимум две независимые причины, по которым 0.5 — плохой выбор по умолчанию в индустриальных задачах:

1. **Издержки ошибок асимметричны.** Пропустить мошенническую транзакцию на 200 000 ₽ (FN) и ошибочно заблокировать легитимную покупку на кофе (FP) — это не одинаковые по цене ошибки. Раздел 3 разбирает это подробно.
2. **Классы несбалансированы.** При сильном дисбалансе (Модуль 3) даже незначительный сдвиг порога от 0.5 может радикально менять соотношение Precision/Recall, и «нейтральная» точка 0.5 зачастую находится в области, где модель либо предсказывает positive крайне редко, либо не предсказывает вовсе (если распределение скоров сильно смещено к нулю из-за малой доли positive класса в обучающих данных).

### 2.4. Формулировка задачи модуля
Вместо вопроса «сколько ошибок совершает модель», нам нужно ответить на вопрос **«сколько эти ошибки стоят, и какой порог минимизирует эту стоимость»**.

## 3. Асимметричные издержки ошибок (10 мин)

### 3.1. Два типа ошибок — две разные цены
Вспомним Confusion Matrix из Модуля 2. Любая ошибка модели — это либо False Positive, либо False Negative. У каждой — своя, зачастую совершенно разного порядка, цена:

| Тип ошибки | Пример: антифрод | Пример: медицинская диагностика |
|---|---|---|
| **False Positive** (ложная тревога) | Легитимная транзакция заблокирована -> клиент недоволен, звонок в поддержку, стоимость ~5 ₽–50 ₽ (время оператора/отток клиента) | Здоровый пациент направлен на доп. обследование -> лишние расходы на анализы, стресс пациента |
| **False Negative** (пропуск) | Мошенническая транзакция пропущена -> банк теряет сумму транзакции целиком, например 500 $ | Больной пациент не выявлен -> отложенное лечение, в худшем случае — цена человеческой жизни |

### 3.2. Почему это меняет всё
Если $C_{FN} \gg C_{FP}$ (пропуск дороже ложной тревоги на порядки), то экономически рационально **сознательно увеличивать число ложных тревог**, чтобы поймать больше реальных positive объектов — то есть сдвигать порог **вниз** относительно 0.5, жертвуя Precision ради Recall. Именно такая логика уже встречалась в проекте FraudGuard (см. `список_проектов.txt`): порог там оказался около 0.12, а не 0.5, именно из-за асимметрии цены пропуска мошенничества и цены ручной проверки оператором.

### 3.3. Формализация как числовых констант

Введем:
- $C_{FP}$ — денежная стоимость одной ошибки False Positive.
- $C_{FN}$ — денежная стоимость одной ошибки False Negative.

В этом модуле мы сначала рассматриваем случай, когда обе константы фиксированы и одинаковы для всех объектов (упрощение). Раздел 7 разбирает более реалистичный случай, когда $C_{FN}$ зависит от конкретного объекта (например, от суммы транзакции).

## 4. Матрица бизнес-потерь: формализация Total Cost (15 мин)

### 4.1. Функция суммарных потерь
Для данного порога $t$ модель дает конкретные значения $FP(t)$ и $FN(t)$ (числа ложных тревог и пропусков при этом пороге). Суммарные потери:
$$TotalCost(t) = FP(t) \cdot C_{FP} + FN(t) \cdot C_{FN}$$

**Задача:** найти
$$t^* = \arg\min_{t \in [0, 1]} TotalCost(t)$$

### 4.2. Как ведут себя $FP(t)$ и $FN(t)$ при изменении порога

Это уже разбиралось в Модуле 2 (раздел 4.5) в терминах Precision/Recall, здесь — то же самое в терминах абсолютных счетчиков:
- При росте порога $t$ модель предсказывает positive для всё меньшего числа объектов => $FP(t)$ **монотонно не возрастает** (ложных тревог становится меньше или столько же), а $FN(t)$ **монотонно не убывает** (пропущенных positive объектов становится больше или столько же).
- Оба крайних случая: при $t=0$ (все объекты — positive) $FN(0) = 0$, а $FP(0)$ равен всему числу negative объектов в выборке; при $t=1$ (ни один объект не positive) $FP(1) = 0$, а $FN(1)$ равен всему числу positive объектов.

Отсюда: $TotalCost(t)$ — это сумма одной невозрастающей и одной неубывающей функции, взвешенных разными константами. Такая сумма не обязана быть строго выпуклой (в общем случае, на реальных данных, могут быть локальные плато и не единственная точка минимума), но обычно имеет один достаточно выраженный глобальный минимум где-то между 0 и 1 — тем более выраженный, чем сильнее асимметрия $C_{FP}$ и $C_{FN}$.

### 4.3. Числовой пример «в лоб»: 20 объектов

Возьмем игрушечный датасет: 6 positive объектов и 14 negative, с такими предсказанными скорами:

- **Positive (label=1):** 0.95, 0.85, 0.70, 0.60, 0.55, 0.30
- **Negative (label=0):** 0.90, 0.65, 0.50, 0.40, 0.35, 0.32, 0.25, 0.20, 0.15, 0.10, 0.08, 0.05, 0.03, 0.02

Пусть $C_{FP} = 5$ (условных единиц, например $), $C_{FN} = 500$.

Посчитаем $TotalCost(t)$ вручную для нескольких порогов (объект считается predicted positive, если его скор $\geq t$):

| $t$ | FP | FN | TotalCost |
|---|---|---|---|
| 0.00 | 14 | 0 | $14 \times 5 + 0 \times 500 = 70$ |
| 0.05 | 12 | 0 | $60$ |
| 0.10 | 10 | 0 | $50$ |
| 0.15 | 9 | 0 | $45$ |
| 0.20 | 8 | 0 | $40$ |
| 0.25 | 7 | 0 | $35$ |
| **0.26** | **6** | **0** | **$30$ <- минимум** |
| 0.30 | 6 | 0 | $30$ (то же плато) |
| 0.31 | 6 | 1 | $6 \times 5 + 1 \times 500 = 530$ |
| 0.50 | 3 | 1 | $515$ |
| 0.70 | 1 | 3 | $1505$ |
| 0.90 | 1 | 5 | $2505$ |
| 1.00 | 0 | 6 | $3000$ |

### 4.4. Что показывает эта таблица
1. **От $t=0$ до $t \approx 0.26$** потери плавно убывают: мы постепенно избавляемся от ложных тревог, ничего не теряя в Recall (все 6 positive объектов пока еще имеют скор выше текущего порога).
2. **На отрезке $t \in (0.25,\ 0.30]$** потери выходят на плато $= 30$: в этом промежутке скоров попросту нет ни одного объекта (ни positive, ни negative), поэтому множество predicted positive не меняется, и стоимость остается неизменной. Именно на этом плато находится истинный оптимум.
3. **При $t = 0.31$** происходит резкий скачок потерь с 30 до 530: порог впервые «перепрыгивает» через скор реального positive объекта (0.30), тот становится False Negative, и единственная такая ошибка обходится в $500$ — дороже, чем полное отсутствие ложных тревог могло сэкономить.
4. Дальнейший рост порога только ухудшает ситуацию: каждый следующий пропущенный positive объект добавляет еще 500 к потерям, в то время как экономия на FP исчисляется единицами.

Этот пример наглядно показывает главную закономерность: **при сильно асимметричных издержках ($C_{FN} \gg C_{FP}$) оптимальный порог систематически смещается вниз, в сторону более мягкого решения, жертвуя Precision ради того, чтобы не терять дорогостоящие positive объекты.**

## 5. Аналитическая формула порога по Байесу — и почему она не заменяет сканирование (15 мин)

### 5.1. Вывод формулы
Допустим, скор модели $p$ — это **действительно откалиброванная** апостериорная вероятность: $p = P(y=1 \mid x)$ в точном статистическом смысле (то есть среди всех объектов, которым модель присвоила скор $p$, доля реальных positive равна именно $p$; подробнее — Модуль 8).

Для конкретного объекта с вероятностью $p$ рассмотрим два возможных решения и их ожидаемую стоимость:
- **Решение «positive»:** ошибка происходит, если объект на самом деле negative (вероятность этого $1-p$), и тогда мы платим $C_{FP}$. Ожидаемая стоимость решения: $E[\text{cost} \mid \text{positive}] = (1-p) \cdot C_{FP}$.
- **Решение «negative»:** ошибка происходит, если объект на самом деле positive (вероятность $p$), и тогда мы платим $C_{FN}$. Ожидаемая стоимость: $E[\text{cost} \mid \text{negative}] = p \cdot C_{FN}$.

Рационально выбрать «positive» тогда и только тогда, когда его ожидаемая стоимость меньше:
$$(1-p) \cdot C_{FP} < p \cdot C_{FN}$$
$$C_{FP} - p \cdot C_{FP} < p \cdot C_{FN}$$
$$C_{FP} < p \cdot (C_{FN} + C_{FP})$$
$$p > \frac{C_{FP}}{C_{FN} + C_{FP}}$$

Получаем **байесовский оптимальный порог**:
$$t^* = \frac{C_{FP}}{C_{FN} + C_{FP}}$$

### 5.2. Проверка: симметричный случай
Если $C_{FP} = C_{FN}$ (ошибки равноценны), формула дает $t^* = \dfrac{C_{FP}}{2 C_{FP}} = 0.5$ — ровно то значение по умолчанию, которое обсуждалось в разделе 2. Формула согласуется с уже известным частным случаем — хороший знак, что вывод корректен.

### 5.3. Применение к числам из раздела 4
$C_{FP} = 5$, $C_{FN} = 500$:
$$t^* = \frac{5}{500 + 5} = \frac{5}{505} \approx 0.0099$$

Формула утверждает, что оптимальный порог — **около 0.01**, практически «предсказывай positive при малейшем подозрении». Но в разделе 4.3 численное сканирование того же самого игрушечного датасета с теми же $C_{FP}, C_{FN}$ дало оптимум **около 0.26–0.30** — на порядок выше!

### 5.4. Почему аналитическая формула не решает практику
Это расхождение — не ошибка, а важный практический урок. Формула из раздела 5.1 верна **только если** скор $p$ — точная калиброванная вероятность. В игрушечном примере раздела 4 значения 0.95, 0.85 и т.д. — это просто **скоры** (баллы модели), не гарантированно являющиеся истинными вероятностями. То же самое верно почти для всех реальных моделей: логистическая регрессия дает приблизительно откалиброванные вероятности при достаточном объеме данных, а вот градиентный бустинг (LightGBM, XGBoost) систематически **искажает** вероятности, сжимая их к 0 и 1 (это будет подробно разобрано в Модуле 8) — то есть скор 0.95 от LightGBM вовсе не означает «95 % объектов с таким скором реально positive».

**Следствие:** формула $t^* = C_{FP}/(C_{FN}+C_{FP})$ полезна как быстрая интуитивная оценка порядка величины и как теоретический ориентир, но в продакшене на нее нельзя полагаться без предварительной калибровки модели. **Численное сканирование порогов (раздел 6) не делает никаких предположений о калибровке — оно работает напрямую с тем, что реально происходит с Confusion Matrix при каждом пороге, и поэтому остается универсальным и надежным методом независимо от того, откалибрована модель или нет.** Именно поэтому индустриальный стандарт — сканирование, а не подстановка в формулу.

## 6. Алгоритм сканирования порогов: наивный и эффективный (15 мин)

### 6.1. Наивный алгоритм (соответствует формулировке задачи в плане курса)
1. Задать сетку порогов, например `np.arange(0.00, 1.01, 0.01)` — 101 значение.
2. Для каждого порога $t$ из сетки: получить `y_pred = (y_proba >= t)`, посчитать $FP(t)$ и $FN(t)$ полным проходом по всем объектам.
3. Вычислить $TotalCost(t) = FP(t) \cdot C_{FP} + FN(t) \cdot C_{FN}$.
4. Выбрать порог с минимальным $TotalCost$.

**Сложность:** для каждого из $K$ порогов требуется полный проход по $n$ объектам, итого $O(K \cdot n)$. При $K=101$ и $n$ в миллионах строк это уже сотни миллионов операций — не катастрофично, но заметно избыточно.

### 6.2. Эффективный алгоритм: сортировка + накопительные суммы
Ключевое наблюдение: $TotalCost(t)$ — кусочно-постоянная функция, которая может измениться только в точках, равных реальным наблюдаемым скорам объектов (между двумя соседними уникальными скорами ничего не меняется в разбиении на predicted positive/negative). Значит, вместо фиксированной сетки из 101 значения достаточно проверить **ровно $n$ кандидатов** — все уникальные скоры, — и это можно сделать за один проход после сортировки.

**Алгоритм:**
1. Отсортировать объекты по скору по убыванию.
2. Начать с порога выше максимального скора: тогда ни один объект не predicted positive, $FP=0$, $FN = $ общее число positive объектов.
3. Идти по отсортированному списку от наибольшего скора к наименьшему, поочередно «включая» каждый объект во множество predicted positive (порог опускается до его скора):
   - если включаемый объект действительно positive — он был FN, теперь становится TP: $FN \mathrel{-}= 1$;
   - если включаемый объект действительно negative — он был TN, теперь становится FP: $FP \mathrel{+}= 1$.
4. После каждого включения пересчитать $TotalCost$ за $O(1)$ (используя обновленные накопленные $FP$, $FN$, без полного пересчета) и сравнить с текущим минимумом.

**Сложность:** доминирует сортировка, $O(n \log n)$, вместо $O(K \cdot n)$ — асимптотически лучше, и при этом проверяются **все** потенциально значимые точки перегиба функции, а не только точки фиксированной сетки, что гарантирует нахождение истинного, а не приближенного оптимума.

### 6.3. Практический вывод
Для датасетов до нескольких сотен тысяч строк разница между наивным и эффективным подходом малозаметна на практике (доли секунды против еще меньших долей секунды), поэтому в реальной работе часто используют наивный вариант из-за простоты кода и читаемости. Эффективная версия становится оправданной при повторяющемся пересчете порога на очень больших объемах данных (например, ежедневный пересчет порога в мониторинге модели на десятках миллионов транзакций) — тогда разница между $O(101n)$ и $O(n \log n)$ действительно ощутима.

## 7. Переменные издержки: когда $C_{FN}$ зависит от объекта (10 мин)

### 7.1. Ограничение упрощенной модели
До сих пор $C_{FP}$ и $C_{FN}$ считались фиксированными константами, одинаковыми для всех объектов. Это разумное упрощение для многих задач (например, если стоимость ручной проверки оператором действительно фиксирована), но не универсально.

### 7.2. Пример: сумма транзакции как переменная стоимость FN
В антифроде пропуск мошеннической транзакции на 500 ₽ и пропуск мошеннической транзакции на 500 000 ₽ — это принципиально разные по цене ошибки, хотя обе классифицируются одинаково как «False Negative». Более точная формулировка суммарных потерь:
$$TotalCost(t) = FP(t) \cdot C_{FP} + \sum_{i \,:\, FN_i(t)} \text{Amount}_i$$
где сумма берется по всем объектам, которые при пороге $t$ являются False Negative, и вместо фиксированной константы $C_{FN}$ используется их индивидуальная сумма транзакции $\text{Amount}_i$.

Это в точности логика функции `find_optimal_threshold` из проекта FraudGuard (`список_проектов.txt`): `fn_cost = fn_mask.sum() * transaction_amounts[fn_mask].sum()` — там стоимость пропуска считается не как число ошибок, умноженное на константу, а как **сумма фактических сумм пропущенных транзакций**.

### 7.3. Как это меняет алгоритм
Изменение минимально: вместо `fn * cost_fn` нужно посчитать `transaction_amounts[fn_mask].sum()` на каждом шаге. Наивный алгоритм (раздел 6.1) адаптируется тривиально — просто заменяется формула стоимости внутри цикла. Эффективный алгоритм (раздел 6.2) тоже адаптируется естественно: вместо счетчика $FN \mathrel{-}=1$ используется накопительная сумма $\text{FN\_cost} \mathrel{-}= \text{Amount}_i$ при включении $i$-го объекта.

### 7.4. Почему это важнее, чем кажется
С переменной стоимостью аналитическая байесовская формула из раздела 5 в общем виде **не работает** — она предполагает единую константу $C_{FN}$ для решения о каждом объекте. При переменной стоимости решение по каждому объекту в принципе должно приниматься с учетом именно его индивидуальной $C_{FN,i}$ — то есть теоретически оптимальный порог **разный для каждого объекта** ($t^*_i = C_{FP}/(C_{FN,i}+C_{FP})$, что для больших транзакций дает очень низкий индивидуальный порог, а для мелких — более высокий). Такой персонализированный порог редко реализуют в проде из-за сложности, но само понимание того, почему единый порог — это компромисс, важно для собеседования.

## 8. Практика: `find_optimal_threshold` (30 мин)

### 8.1. Постановка задачи
Реализовать функцию `find_optimal_threshold(y_true, y_proba, cost_fn, cost_fp)` (наивная версия по спецификации плана курса), проверить её на игрушечном датасете из раздела 4.3, затем применить на реалистичном синтетическом датасете, визуализировать кривую Total Cost и сравнить с потерями при пороге по умолчанию 0.5. Дополнительно — реализовать эффективную версию (раздел 6.2) и сверить результаты.

### 8.2. Код: наивная реализация

In [ ]:
import numpy as np

def find_optimal_threshold(y_true, y_proba, cost_fn, cost_fp, thresholds=None):
    """
    Находит порог классификации, минимизирующий суммарные финансовые потери.

    Параметры
    ---------
    y_true : array-like, shape (n_samples,)
        Истинные метки классов (0 или 1).
    y_proba : array-like, shape (n_samples,)
        Предсказанные вероятности положительного класса.
    cost_fn : float
        Стоимость одной ошибки False Negative (пропуск positive объекта).
    cost_fp : float
        Стоимость одной ошибки False Positive (ложная тревога).
    thresholds : array-like, optional
        Сетка порогов для перебора. По умолчанию — 0.00 до 1.00 с шагом 0.01.

    Возвращает
    ----------
    dict:
        'best_threshold' — порог с минимальными потерями
        'best_cost'      — величина минимальных потерь
        'thresholds'     — все проверенные пороги (для графика)
        'costs'          — потери для каждого порога (для графика)
    """
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    if thresholds is None:
        thresholds = np.arange(0.00, 1.01, 0.01)

    costs = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        total_cost = fp * cost_fp + fn * cost_fn
        costs.append(total_cost)

    costs = np.array(costs)
    best_idx = np.argmin(costs)

    return {
        'best_threshold': thresholds[best_idx],
        'best_cost': costs[best_idx],
        'thresholds': thresholds,
        'costs': costs
    }

### 8.3. Проверка на игрушечном датасете из раздела 4.3

In [ ]:
y_true_toy = np.array([1, 1, 1, 1, 1, 1] + [0] * 14)
y_proba_toy = np.array(
    [0.95, 0.85, 0.70, 0.60, 0.55, 0.30] +                                   # positive
    [0.90, 0.65, 0.50, 0.40, 0.35, 0.32, 0.25, 0.20, 0.15, 0.10, 0.08, 0.05, 0.03, 0.02]  # negative
)

result_toy = find_optimal_threshold(y_true_toy, y_proba_toy, cost_fn=500, cost_fp=5)

print(f"Оптимальный порог: {result_toy['best_threshold']:.2f}")
print(f"Минимальные потери: {result_toy['best_cost']:.0f}")
# Ожидается: порог где-то в диапазоне 0.26-0.30, потери = 30 — сверка с ручным расчетом из раздела 4.3

### 8.4. Эффективная реализация (сортировка + накопительные суммы)

In [ ]:
def find_optimal_threshold_fast(y_true, y_proba, cost_fn, cost_fp):
    """
    Эффективная версия: O(n log n) вместо O(n * n_thresholds).
    Проверяет ВСЕ уникальные наблюдаемые скоры как кандидаты в пороги —
    гарантированно находит точный оптимум, а не приближение по сетке.
    """
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)

    order = np.argsort(-y_proba)          # сортировка по убыванию скора
    y_true_sorted = y_true[order]
    proba_sorted = y_proba[order]

    # Стартовая точка: порог выше максимального скора — ничего не предсказываем positive
    fp = 0
    fn = int(np.sum(y_true == 1))

    best_cost = fp * cost_fp + fn * cost_fn
    best_threshold = 1.0 + 1e-9

    thresholds_path = [best_threshold]
    costs_path = [best_cost]

    for i in range(n):
        if y_true_sorted[i] == 1:
            fn -= 1   # был FN, стал TP
        else:
            fp += 1   # был TN, стал FP

        cost = fp * cost_fp + fn * cost_fn
        thresholds_path.append(proba_sorted[i])
        costs_path.append(cost)

        if cost < best_cost:
            best_cost = cost
            best_threshold = proba_sorted[i]

    return {
        'best_threshold': best_threshold,
        'best_cost': best_cost,
        'thresholds': np.array(thresholds_path),
        'costs': np.array(costs_path)
    }

# Сверка результатов на игрушечном датасете
result_toy_fast = find_optimal_threshold_fast(y_true_toy, y_proba_toy, cost_fn=500, cost_fp=5)
print(f"Наивная версия:     порог={result_toy['best_threshold']:.4f}, потери={result_toy['best_cost']:.0f}")
print(f"Эффективная версия: порог={result_toy_fast['best_threshold']:.4f}, потери={result_toy_fast['best_cost']:.0f}")
# best_cost должен совпасть точно (30);
# best_threshold может немного отличаться — эффективная версия найдет точку 0.30 (реальный скор),
# наивная — любую точку сетки 0.01 внутри плато [0.26; 0.30]

### 8.5. Применение на реалистичном датасете + визуализация

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# 1. Датасет с умеренным дисбалансом (5% positive)
X, y = make_classification(
    n_samples=20_000, n_features=20, n_informative=10,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 2. Издержки: пропуск в 100 раз дороже ложной тревоги
COST_FP = 5
COST_FN = 500

result = find_optimal_threshold(y_test, y_proba, cost_fn=COST_FN, cost_fp=COST_FP)

# 3. Потери при пороге по умолчанию (0.5) для сравнения
y_pred_default = (y_proba >= 0.5).astype(int)
fp_default = np.sum((y_pred_default == 1) & (y_test == 0))
fn_default = np.sum((y_pred_default == 0) & (y_test == 1))
cost_default = fp_default * COST_FP + fn_default * COST_FN

print(f"Оптимальный порог:              {result['best_threshold']:.2f}")
print(f"Потери при оптимальном пороге:  ${result['best_cost']:,.0f}")
print(f"Потери при пороге 0.5:          ${cost_default:,.0f}")
print(f"Экономия от смены порога:       ${cost_default - result['best_cost']:,.0f} "
      f"({(1 - result['best_cost'] / cost_default):.1%})")

# 4. Визуализация кривой Total Cost
plt.figure(figsize=(10, 6))
plt.plot(result['thresholds'], result['costs'], linewidth=2)
plt.axvline(result['best_threshold'], color='green', linestyle='--',
            label=f"Оптимальный порог = {result['best_threshold']:.2f}")
plt.axvline(0.5, color='red', linestyle='--', label='Порог по умолчанию = 0.50')
plt.xlabel('Порог классификации')
plt.ylabel('Суммарные потери, $')
plt.title('Total Cost в зависимости от порога')
plt.legend()
plt.grid(True)
plt.show()

### 8.6. Что должно получиться и на что смотреть
- Оптимальный порог окажется значительно ниже 0.5 (обычно в диапазоне 0.05–0.20 при таком соотношении издержек) — модель должна «охотнее» предсказывать positive, чем при пороге по умолчанию.
- Потери при пороге 0.5 будут заметно выше, чем при оптимальном — это прямая, измеримая в деньгах демонстрация того, зачем нужен весь этот модуль: неверно выбранный порог напрямую конвертируется в финансовые потери.
- На графике кривая `TotalCost(t)` типично имеет форму, похожую на широкую букву U (или несимметричную чашу): резкий рост при приближении к $t=1$ (стремительно растет FN-компонент, вклад которого дороже) и более пологий рост при приближении к $t=0$ (растет FP-компонент, но он дешевле).
- Обе реализации — наивная и эффективная — должны дать совпадающее (или крайне близкое, в пределах разрешения сетки 0.01) значение `best_cost`.

## 9. Итоги модуля (5 мин)

### Ключевые тезисы
1. Порог 0.5 — техническая конвенция, оптимальная только при откалиброванной модели и равной стоимости ошибок обоих типов; ни одно из этих условий не гарантировано в реальных задачах.
2. Формализация решения: $TotalCost(t) = FP(t) \cdot C_{FP} + FN(t) \cdot C_{FN}$, оптимальный порог — точка минимума этой функции.
3. Байесовский аналитический порог $t^* = C_{FP}/(C_{FN}+C_{FP})$ дает верный ответ **только** при точно откалиброванных вероятностях — условие, которое почти никогда не выполняется автоматически, особенно для градиентного бустинга.
4. Численное сканирование порогов не требует калибровки и остается надежным методом в любой ситуации; наивная версия — $O(K \cdot n)$, эффективная версия через сортировку и накопительные суммы — $O(n \log n)$ и находит точный, а не приближенный по сетке оптимум.
5. При переменной стоимости FN (например, зависящей от суммы транзакции) аналитическая формула теряет единственное решение, а численный алгоритм адаптируется тривиально — заменой суммы констант на сумму индивидуальных издержек.

### Контрольные вопросы
- При каких двух условиях порог 0.5 действительно является оптимальным решением?
- Выведите формулу байесовского порога $t^*$ из сравнения ожидаемых потерь двух решений для объекта с вероятностью $p$.
- Почему в разделе 5 численный и аналитический подходы дали разный ответ на одних и тех же данных, и какой из них считать «более правильным» в общем случае?
- Почему эффективный алгоритм поиска порога использует именно уникальные наблюдаемые скоры в качестве кандидатов, а не фиксированную сетку с шагом 0.01?
- Как изменится функция `find_optimal_threshold`, если стоимость FN зависит от суммы транзакции конкретного объекта, а не является константой?

### Что дальше
В следующем модуле мы разберем техники балансировки классов (`class_weight`, SMOTE) и покажем, как эти техники при неаккуратном применении искажают именно те вероятности, на которых основан весь сегодняшний модуль — сдвигая скоры модели и делая поиск порога по сырым `predict_proba` ненадежным без дополнительной осторожности.